# Phase 4b: 6 more pushback templates (hedged vs blunt) + large free-text check (7B, bf16)

**Same GPU as Phase 4 (A100)**, then `Runtime → Run all`. Needs the Phase 4 outputs on Drive.

1. GPU check, **backup** of the Phase 4 files (`*_v1_3templates.csv`), then a **smoke test** of every script.
2. **Behavior** on all 9 templates (≈42k prompts; the baseline filter is reused from Phase 4's checkpoint).
3. **Replication check**: the 3 original templates should reproduce Phase 4's flips almost exactly.
4. **Stats**: the pre-registered analysis (3 original templates) + hedged vs blunt family analysis (all 9).
5. **Free text** on 120 questions × 13 conditions × 9 templates (≈14k replies), checkpointed, with tests on the free-reply flips.
6. Summary to paste back, then auto-disconnect. If Colab disconnects, `Run all` again: every long step resumes.

In [ ]:
CFG = "configs/colab.yaml"   # same config as Phase 4
AUTO_DISCONNECT = True

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops).
    Splits only on real newlines and keeps the text after the last carriage return, so tqdm
    progress bars collapse to one line instead of flooding the notebook."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for raw in p.stdout:
            line = raw.decode('utf-8', 'replace').rstrip('\n').split('\r')[-1]
            if line.strip():
                print(line, flush=True)
                f.write(line + '\n')
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check, backup of Phase 4 outputs, smoke test

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

import os, shutil, time
OUT = '/content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct'
assert os.path.exists(f'{OUT}/behavior.csv'), 'Phase 4 behavior.csv not found on Drive'
import pandas as pd
if pd.read_csv(f'{OUT}/behavior.csv', usecols=['template']).template.nunique() == 3:
    for f in ['behavior', 'stats_rates', 'stats_tests', 'stats_template_tests', 'stats_template_interaction',
              'stats_template_by_pair', 'freetext', 'freetext_sample']:
        src_f, dst_f = f'{OUT}/{f}.csv', f'{OUT}/{f}_v1_3templates.csv'
        if os.path.exists(src_f) and not os.path.exists(dst_f):
            shutil.copy(src_f, dst_f)
            print('backed up', dst_f)
assert os.path.exists(f'{OUT}/behavior_v1_3templates.csv'), 'no Phase 4 backup found'

SMOKE = {'VNP_OUTPUT_DIR': '/content/smoke'}
shutil.rmtree('/content/smoke', ignore_errors=True)
t0 = time.time()
run(f'python -m vnpronouns.run_behavior --config {CFG} --pool 2 --no-filter', '/content/log_smoke.txt', SMOKE)
run(f'python -m vnpronouns.stats --config {CFG} --n-boot 200', '/content/log_smoke.txt', SMOKE)
run(f'python -m vnpronouns.freetext --config {CFG} --n-questions 1', '/content/log_smoke.txt', SMOKE)
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')

## 2. Behavior on all 9 templates

In [ ]:
run(f'python -m vnpronouns.run_behavior --config {CFG}', '/content/log_behavior.txt')

## 3. Replication check: original 3 templates vs Phase 4

In [ ]:
old = pd.read_csv(f'{OUT}/behavior_v1_3templates.csv')
new = pd.read_csv(f'{OUT}/behavior.csv')
key = ['qid', 'condition', 'template']
m = old[key + ['flip', 'p_correct']].merge(new[key + ['flip', 'p_correct']], on=key, suffixes=('_v1', '_new'))
rep = (f'REPLICATION: {len(m)}/{len(old)} Phase 4 prompts matched; identical flip on {(m.flip_v1 == m.flip_new).mean():.2%}; '
       f'max |dP(correct)| {(m.p_correct_v1 - m.p_correct_new).abs().max():.4f}; '
       f'flip rate v1 {m.flip_v1.mean():.4f} vs new {m.flip_new.mean():.4f}')
print(rep)
open('/content/log_replication.txt', 'w').write(rep + '\n')

## 4. Statistics

In [ ]:
!rm -f /content/log_stats.txt
run(f'python -m vnpronouns.stats --config {CFG}', '/content/log_stats.txt')

## 5. Free text: 120 questions × 13 conditions × 9 templates

In [ ]:
!rm -f /content/log_freetext.txt
run(f'python -m vnpronouns.freetext --config {CFG} --n-questions 120', '/content/log_freetext.txt')

## 6. Summary to paste back to Claude

In [ ]:
import re
def clean(path, last_run_only=False):
    lines = open(path, encoding='utf-8').read().splitlines()
    if last_run_only:
        starts = [i for i, l in enumerate(lines) if l.startswith('baseline:')]
        lines = lines[starts[-1]:] if starts else lines
    return '\n'.join(l.rstrip() for l in lines
                     if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk |^\$ ', l))
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p, last in [('/content/log_behavior.txt', True), ('/content/log_replication.txt', False),
                ('/content/log_stats.txt', False), ('/content/log_freetext.txt', False)]:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p, last))
!ls -la /content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct/

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()